# Ridge Score Walk-Forward - 1h Candles

Current hypothesis: `momentum_only` is the alpha baseline. Roll-impact terms are retained here only as a historical comparison and as potential execution/risk diagnostics; they must beat the momentum-only baseline out-of-sample after costs before being considered for deployment.

Score family under test:

`score = beta_mom * z_mom + beta_roll * z_low_roll + beta_interaction * z_mom * z_low_roll`

The notebook remains useful for comparing score families, but the V1/V2 refit policy experiment is the current deployment-style test.


## Protocol

- Target: 24h forward return from 1h candles.
- Fit: ridge regression on cross-sectional z-scored forward returns inside each IS window.
- Selection: choose ridge alpha by IS mean Spearman IC for each model family.
- Evaluation: freeze the selected weights and evaluate OS Spearman/Pearson IC plus top-K net returns.
- Window: 3 months IS, 2 months OS, step 1 month.
- Data: canonical 30-month feature file `data/features/live_1h_features_30m.csv`.

Old short-sample outputs were cleared because they are stale and should not be mixed with the current refit-policy hypothesis.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import HTML, display

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = next(parent for parent in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents) if (parent / "bot").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from bot.forward_ic import add_forward_return, information_coefficient  # noqa: E402

FEATURE_PATH = REPO_ROOT / "data/features/live_1h_features_30m.csv"
OUTPUT_DIR = REPO_ROOT / "notebooks/microstructure"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZON = 24
IS_MONTHS = 3
OS_MONTHS = 2
STEP_MONTHS = 1
RIDGE_ALPHAS = (0.1, 1.0, 10.0, 100.0)
TOP_KS = (3, 5)
COST_BPS = 20
MIN_ASSETS_PER_TIMESTAMP = 8
MIN_IC_ASSETS = 8

BASE_SIGNAL_SPECS = (
    {
        "name": "momentum",
        "source_col": "vol_adj_momentum_24",
        "transform": "identity",
    },
    {
        "name": "low_roll_impact",
        "source_col": "roll_impact",
        "transform": "negative",
    },
)
MODEL_SPECS = (
    {
        "name": "momentum_only",
        "terms": ("z_momentum",),
    },
    {
        "name": "momentum_plus_roll",
        "terms": ("z_momentum", "z_low_roll_impact"),
    },
    {
        "name": "momentum_roll_interaction",
        "terms": ("z_momentum", "z_momentum_x_low_roll_impact"),
    },
    {
        "name": "momentum_plus_roll_plus_interaction",
        "terms": ("z_momentum", "z_low_roll_impact", "z_momentum_x_low_roll_impact"),
    },
)


In [ ]:
def normalize_time(frame: pd.DataFrame, time_col: str = "open_time") -> pd.DataFrame:
    out = frame.copy()
    if pd.api.types.is_numeric_dtype(out[time_col]):
        out["timestamp"] = pd.to_datetime(out[time_col], unit="ms", utc=True)
    else:
        out["timestamp"] = pd.to_datetime(out[time_col], utc=True)
    return out


def cross_sectional_zscore(frame: pd.DataFrame, value_col: str, time_col: str = "open_time") -> pd.Series:
    grouped = frame.groupby(time_col, observed=True)[value_col]
    mean = grouped.transform("mean")
    std = grouped.transform("std").replace(0, np.nan)
    return ((frame[value_col] - mean) / std).replace([np.inf, -np.inf], np.nan)


def load_model_frame() -> pd.DataFrame:
    df = pd.read_csv(FEATURE_PATH)
    df = normalize_time(df)
    target_col = f"forward_return_{HORIZON}"
    if target_col not in df.columns:
        df = add_forward_return(
            df,
            horizon=HORIZON,
            price_col="close",
            target_col=target_col,
            group_col="pair",
            sort_col="open_time",
        )

    for spec in BASE_SIGNAL_SPECS:
        raw_col = f"raw_{spec['name']}"
        if spec["transform"] == "negative":
            df[raw_col] = -df[spec["source_col"]]
        elif spec["transform"] == "identity":
            df[raw_col] = df[spec["source_col"]]
        else:
            raise ValueError(f"Unknown transform: {spec['transform']}")
        df[f"z_{spec['name']}"] = cross_sectional_zscore(df, raw_col)

    df["raw_momentum_x_low_roll_impact"] = df["z_momentum"] * df["z_low_roll_impact"]
    df["z_momentum_x_low_roll_impact"] = cross_sectional_zscore(
        df,
        "raw_momentum_x_low_roll_impact",
    )
    df["target_z"] = cross_sectional_zscore(df, target_col)
    return df.sort_values(["pair", "timestamp"]).reset_index(drop=True)


def make_folds(frame: pd.DataFrame) -> pd.DataFrame:
    start = pd.Timestamp(frame["timestamp"].min())
    end = pd.Timestamp(frame["timestamp"].max())
    rows = []
    fold = 0
    cursor = start
    while True:
        is_start = cursor
        is_end = is_start + pd.DateOffset(months=IS_MONTHS)
        os_start = is_end
        os_end = os_start + pd.DateOffset(months=OS_MONTHS)
        if os_end > end:
            break
        rows.append(
            {
                "fold": fold,
                "is_start": is_start,
                "is_end": is_end,
                "os_start": os_start,
                "os_end": os_end,
            }
        )
        fold += 1
        cursor = cursor + pd.DateOffset(months=STEP_MONTHS)
    if not rows:
        raise ValueError(
            f"Not enough data for {IS_MONTHS}m IS + {OS_MONTHS}m OS. "
            f"Available: {start} to {end}."
        )
    return pd.DataFrame(rows)


def fit_ridge(train: pd.DataFrame, terms: tuple[str, ...], alpha: float) -> np.ndarray:
    valid = train.dropna(subset=[*terms, "target_z"])
    x = valid.loc[:, terms].to_numpy(dtype=float)
    y = valid["target_z"].to_numpy(dtype=float)
    penalty = alpha * np.eye(x.shape[1])
    return np.linalg.pinv(x.T @ x + penalty) @ x.T @ y


In [ ]:
def score_frame(frame: pd.DataFrame, terms: tuple[str, ...], beta: np.ndarray, score_col: str) -> pd.DataFrame:
    out = frame.copy()
    valid = out.loc[:, terms].notna().all(axis=1)
    out[score_col] = np.nan
    out.loc[valid, score_col] = out.loc[valid, terms].to_numpy(dtype=float) @ beta
    return out


def ic_by_time(frame: pd.DataFrame, score_col: str, target_col: str, sample: str) -> pd.DataFrame:
    rows = []
    for open_time, group in frame.dropna(subset=[score_col, target_col]).groupby("open_time", observed=True):
        if len(group) < MIN_IC_ASSETS:
            continue
        if group[score_col].nunique(dropna=True) < 2 or group[target_col].nunique(dropna=True) < 2:
            continue
        pearson, pearson_n = information_coefficient(
            group[score_col],
            group[target_col],
            method="pearson",
            min_periods=MIN_IC_ASSETS,
        )
        spearman, spearman_n = information_coefficient(
            group[score_col],
            group[target_col],
            method="spearman",
            min_periods=MIN_IC_ASSETS,
        )
        rows.append(
            {
                "sample": sample,
                "open_time": open_time,
                "timestamp": group["timestamp"].iloc[0],
                "pearson": pearson,
                "pearson_n": pearson_n,
                "spearman": spearman,
                "spearman_n": spearman_n,
            }
        )
    return pd.DataFrame(rows)


def summarize_ic(ic: pd.DataFrame) -> dict:
    if ic.empty:
        return {
            "periods": 0,
            "mean_pearson": np.nan,
            "mean_spearman": np.nan,
            "median_spearman": np.nan,
            "spearman_hit_rate": np.nan,
        }
    return {
        "periods": int(len(ic)),
        "mean_pearson": float(ic["pearson"].mean()),
        "mean_spearman": float(ic["spearman"].mean()),
        "median_spearman": float(ic["spearman"].median()),
        "spearman_hit_rate": float((ic["spearman"] > 0).mean()),
    }


def topk_period_returns(frame: pd.DataFrame, score_col: str, target_col: str, top_k: int) -> pd.DataFrame:
    valid = frame.dropna(subset=[score_col, target_col]).copy()
    enough_assets = valid.groupby("open_time", observed=True)["pair"].transform("count")
    valid = valid[enough_assets >= MIN_ASSETS_PER_TIMESTAMP]
    valid["score_rank"] = valid.groupby("open_time", observed=True)[score_col].rank(
        ascending=False,
        method="first",
    )
    selected = valid[valid["score_rank"] <= top_k].copy()
    period = selected.groupby(["open_time", "timestamp"], observed=True).agg(
        portfolio_return=(target_col, "mean"),
        selected_count=("pair", "count"),
    )
    period = period.reset_index()
    period["top_k"] = top_k
    period["net_return"] = period["portfolio_return"] - COST_BPS / 10_000
    return period


In [ ]:
def run_walkforward(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    folds = make_folds(frame)
    target_col = f"forward_return_{HORIZON}"
    all_eval_rows = []
    selected_rows = []
    weight_rows = []
    topk_rows = []

    for fold in folds.itertuples(index=False):
        is_mask = (frame["timestamp"] >= fold.is_start) & (frame["timestamp"] < fold.is_end)
        os_mask = (frame["timestamp"] >= fold.os_start) & (frame["timestamp"] < fold.os_end)
        is_frame = frame[is_mask].copy()
        os_frame = frame[os_mask].copy()

        for model in MODEL_SPECS:
            terms = tuple(model["terms"])
            candidates = []
            for alpha in RIDGE_ALPHAS:
                beta = fit_ridge(is_frame, terms, alpha)
                score_col = "ridge_score"
                is_scored = score_frame(is_frame, terms, beta, score_col)
                os_scored = score_frame(os_frame, terms, beta, score_col)
                is_summary = summarize_ic(ic_by_time(is_scored, score_col, target_col, "IS"))
                os_summary = summarize_ic(ic_by_time(os_scored, score_col, target_col, "OS"))
                row = {
                    "fold": fold.fold,
                    "model": model["name"],
                    "alpha": alpha,
                    "is_start": fold.is_start,
                    "is_end": fold.is_end,
                    "os_start": fold.os_start,
                    "os_end": fold.os_end,
                    **{f"is_{k}": v for k, v in is_summary.items()},
                    **{f"os_{k}": v for k, v in os_summary.items()},
                }
                all_eval_rows.append(row)
                candidates.append((row, beta, is_scored, os_scored))

            selected = sorted(
                candidates,
                key=lambda item: (
                    item[0]["is_mean_spearman"],
                    item[0]["is_spearman_hit_rate"],
                    -item[0]["alpha"],
                ),
                reverse=True,
            )[0]
            selected_row, beta, is_scored, os_scored = selected
            selected_rows.append(selected_row | {"selected": True})
            for term, coef in zip(terms, beta, strict=True):
                weight_rows.append(
                    {
                        "fold": fold.fold,
                        "model": model["name"],
                        "alpha": selected_row["alpha"],
                        "term": term,
                        "beta": float(coef),
                    }
                )
            for sample, scored in (("IS", is_scored), ("OS", os_scored)):
                for top_k in TOP_KS:
                    period = topk_period_returns(scored, "ridge_score", target_col, top_k)
                    period["fold"] = fold.fold
                    period["model"] = model["name"]
                    period["alpha"] = selected_row["alpha"]
                    period["sample"] = sample
                    topk_rows.append(period)

    selected_summary = pd.DataFrame(selected_rows)
    all_eval = pd.DataFrame(all_eval_rows)
    weights = pd.DataFrame(weight_rows)
    topk = pd.concat(topk_rows, ignore_index=True) if topk_rows else pd.DataFrame()
    return all_eval, selected_summary, weights, topk


def aggregate_selected(selected_summary: pd.DataFrame, topk: pd.DataFrame) -> pd.DataFrame:
    ic_agg = selected_summary.groupby("model", observed=True).agg(
        folds=("fold", "count"),
        mean_is_spearman=("is_mean_spearman", "mean"),
        mean_os_spearman=("os_mean_spearman", "mean"),
        mean_os_pearson=("os_mean_pearson", "mean"),
        os_spearman_hit_rate=("os_mean_spearman", lambda s: float((s > 0).mean())),
    )
    topk_os = topk[topk["sample"].eq("OS")]
    topk_agg = topk_os.groupby(["model", "top_k"], observed=True).agg(
        os_topk_periods=("net_return", "count"),
        mean_os_net_return=("net_return", "mean"),
        os_topk_hit_rate=("portfolio_return", lambda s: float((s > 0).mean())),
        q05_os_net_return=("net_return", lambda s: s.quantile(0.05)),
    )
    topk_wide = topk_agg.reset_index().pivot(
        index="model",
        columns="top_k",
        values=["mean_os_net_return", "os_topk_hit_rate", "q05_os_net_return"],
    )
    topk_wide.columns = [f"{metric}_top{top_k}" for metric, top_k in topk_wide.columns]
    return ic_agg.join(topk_wide, how="left").reset_index().sort_values(
        "mean_os_spearman",
        ascending=False,
    )


In [ ]:
def svg_bar(
    df: pd.DataFrame,
    title: str,
    label_col: str,
    value_col: str,
    width: int = 980,
    left: int = 360,
    row_h: int = 28,
) -> str:
    rows = df[[label_col, value_col]].dropna().copy().sort_values(value_col)
    height = 54 + row_h * len(rows)
    plot_w = width - left - 90
    max_abs = max(float(rows[value_col].abs().max()), 0.001)
    zero_x = left + plot_w / 2
    scale = (plot_w / 2) / max_abs
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}"><rect width="100%" height="100%" fill="#fff"/>'
    ]
    parts.append(f'<text x="12" y="24" font-family="Arial" font-size="16" font-weight="700">{title}</text>')
    parts.append(f'<line x1="{zero_x:.1f}" y1="38" x2="{zero_x:.1f}" y2="{height - 10}" stroke="#555"/>')
    for i, row in enumerate(rows.itertuples(index=False)):
        label = str(getattr(row, label_col)).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        val = float(getattr(row, value_col))
        y = 46 + i * row_h
        bar_w = abs(val) * scale
        x = zero_x if val >= 0 else zero_x - bar_w
        color = "#2f7d32" if val >= 0 else "#b23a3a"
        parts.append(f'<text x="12" y="{y + 15}" font-family="Arial" font-size="12">{label}</text>')
        parts.append(f'<rect x="{x:.1f}" y="{y}" width="{bar_w:.1f}" height="18" fill="{color}" opacity="0.84"/>')
        tx = zero_x + bar_w + 6 if val >= 0 else zero_x - bar_w - 70
        parts.append(f'<text x="{tx:.1f}" y="{y + 15}" font-family="Arial" font-size="12">{val:+.4f}</text>')
    parts.append("</svg>")
    return "".join(parts)


In [ ]:
frame = load_model_frame()
folds = make_folds(frame)
all_eval, selected_summary, weights, topk_returns = run_walkforward(frame)
aggregate_summary = aggregate_selected(selected_summary, topk_returns)

all_eval.to_csv(OUTPUT_DIR / "ridge_score_walkforward_1h_all_alpha_eval.csv", index=False)
selected_summary.to_csv(OUTPUT_DIR / "ridge_score_walkforward_1h_selected_summary.csv", index=False)
weights.to_csv(OUTPUT_DIR / "ridge_score_walkforward_1h_weights.csv", index=False)
topk_returns.to_csv(OUTPUT_DIR / "ridge_score_walkforward_1h_topk_returns.csv", index=False)
aggregate_summary.to_csv(OUTPUT_DIR / "ridge_score_walkforward_1h_aggregate_summary.csv", index=False)

metadata = {
    "source_features": str(FEATURE_PATH),
    "horizon": HORIZON,
    "is_months": IS_MONTHS,
    "os_months": OS_MONTHS,
    "step_months": STEP_MONTHS,
    "ridge_alphas": list(RIDGE_ALPHAS),
    "top_ks": list(TOP_KS),
    "cost_bps": COST_BPS,
    "rows": int(len(frame)),
    "pairs": int(frame["pair"].nunique()),
    "available_start": str(frame["timestamp"].min()),
    "available_end": str(frame["timestamp"].max()),
    "fold_count": int(len(folds)),
    "base_signal_specs": list(BASE_SIGNAL_SPECS),
    "model_specs": list(MODEL_SPECS),
}
(OUTPUT_DIR / "ridge_score_walkforward_1h_metadata.json").write_text(
    json.dumps(metadata, indent=2, sort_keys=True, default=str),
    encoding="utf-8",
)
metadata


## Folds


In [ ]:
fold_display = folds.copy()
for col in ("is_start", "is_end", "os_start", "os_end"):
    fold_display[col] = fold_display[col].astype(str)
fold_display


## Aggregate OS Results


In [ ]:
aggregate_summary


In [ ]:
aggregate_plot = aggregate_summary.copy()
aggregate_plot["label"] = aggregate_plot["model"]
display(HTML(svg_bar(aggregate_plot, "Mean OS Spearman IC by selected ridge score", "label", "mean_os_spearman")))


In [ ]:
aggregate_plot = aggregate_summary.copy()
aggregate_plot["label"] = aggregate_plot["model"]
display(HTML(svg_bar(aggregate_plot, "Mean OS net top-3 return by selected ridge score", "label", "mean_os_net_return_top3")))


## Selected Weights


In [ ]:
weights.sort_values(['model', 'fold', 'term'])


## Read

        The only disciplined comparison is OS performance after IS alpha selection. If roll-aware model families do not beat `momentum_only` on OS IC and top-K net return, roll impact has not added useful incremental signal in this setup.
